# STC tv - التنبؤ بعدد المشاهدات (Jawwy)
بناء نموذج تنبؤي سهل وبسيط يمكّن المسؤولين من التنبؤ بإجمالي ساعات المشاهدة المتوقعة خلال الشهرين القادمين، وتحديد أوقات الذروة المحتملة.

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd               # data structures and data analysis tools
import numpy as np                # fast mathematical computation on arrays and matrices
from sklearn.linear_model import LinearRegression          # simple, interpretable forecasting model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   # model evaluation metrics

# Jawwy dataset
The dataset includes total watching hours for customers per day.

You are required to work on predicting the forecast for the watching hours.

In [ ]:
# Upload the dataset file "stc_TV_Data_Set_T2.xlsx" to the Colab session (Files panel on the left) before running this cell
dataframe = pd.read_excel("stc_TV_Data_Set_T2.xlsx", index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape (rows, columns)
print("Shape (rows, columns):", dataframe.shape)

In [ ]:
# display the first 5 rows
dataframe.head()

In [ ]:
# check the data types of each column
dataframe.dtypes

In [ ]:
# describe the numeric values in the dataset
dataframe.describe()

In [ ]:
# check if any column has null values
dataframe.isnull().any()

## ملاحظة حول البيانات
البيانات المتوفرة هي إجمالي ساعات المشاهدة (`Total_watch_time_in_houres`) لكل يوم، تغطي الفترة من 2018-01-01 إلى 2018-04-30، وتشمل أيام الأسبوع من الاثنين إلى الجمعة فقط (بدون عطلة نهاية الأسبوع السبت/الأحد ضمن البيانات). هذا يعني أن أي تنبؤ مستقبلي يجب أن يتبع نفس النمط (أيام عمل فقط) ليكون متوافقًا مع طريقة جمع البيانات الأصلية.

In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# show the full historical time series
fig = px.line(dataframe, x='date_', y='Total_watch_time_in_houres',
              title='Daily total watch time (hours) - history')
fig.show()

In [ ]:
# study the weekday pattern: average watch hours by day of week
df = dataframe.copy()
df['weekday_name'] = df['date_'].dt.day_name()
wd_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']
wd_avg = df.groupby('weekday_name')['Total_watch_time_in_houres'].mean().reindex(wd_order)

fig = px.bar(wd_avg.reset_index(), x='weekday_name', y='Total_watch_time_in_houres',
             title='Average watch hours by weekday (historical)',
             labels={'weekday_name':'Weekday','Total_watch_time_in_houres':'Avg watch hours'})
fig.show()
wd_avg

# بناء نموذج التنبؤ
نختار نموذج **انحدار خطي (Linear Regression)** لبساطته وسهولة تفسيره للمسؤولين: يفترض النموذج أن إجمالي ساعات المشاهدة اليومية = اتجاه عام مع الزمن (Trend) + تأثير ثابت لكل يوم من أيام الأسبوع (Weekly Seasonality). هذا يكفي لرصد الاتجاه العام (تصاعد/تراجع) وأوقات الذروة الأسبوعية المتكررة، وهو نموذج يمكن لأي مسؤول غير تقني فهمه وتفسيره بسهولة (لكل متغير معامل واحد واضح المعنى).

In [ ]:
# Feature engineering: a simple time trend (t) + weekday dummy variables
model_df = dataframe.copy().sort_values('date_').reset_index(drop=True)
model_df['weekday'] = model_df['date_'].dt.dayofweek      # 0=Monday ... 4=Friday
model_df['t'] = np.arange(len(model_df))                  # simple day index capturing the overall trend

weekday_dummies = pd.get_dummies(model_df['weekday'], prefix='wd', drop_first=True)
X = pd.concat([model_df[['t']], weekday_dummies], axis=1)
y = model_df['Total_watch_time_in_houres']
X.head()

In [ ]:
# Validate the model honestly: hold out the last 10 days (~2 weeks) as a test set
n_test = 10
X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]

validation_model = LinearRegression().fit(X_train, y_train)
y_pred_test = validation_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_test)
rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
train_r2 = r2_score(y_train, validation_model.predict(X_train))

print(f"MAE on holdout:  {mae:.2f} hours")
print(f"RMSE on holdout: {rmse:.2f} hours")
print(f"R2 on training data: {train_r2:.3f}")

In [ ]:
# Once we are satisfied with the validation results, refit the model on ALL available
# historical data to make the most informed forecast going forward
model = LinearRegression().fit(X, y)

coef_table = pd.Series(model.coef_, index=X.columns, name='effect_on_watch_hours')
print("Intercept (baseline, Monday, t=0):", round(model.intercept_, 2))
coef_table.round(2)

**كيف نقرأ المعاملات (Coefficients)؟** معامل `t` السالب يعني أن إجمالي ساعات المشاهدة اليومية يتراجع تدريجيًا مع الوقت (اتجاه عام نزولي خلال الفترة المرصودة). بقية المعاملات (`wd_1` إلى `wd_4`) تمثل الفرق عن يوم الإثنين (وهو خط الأساس Baseline)؛ فمثلًا قيمة موجبة ليوم الجمعة تعني أن المشاهدة في يوم الجمعة أعلى من الإثنين بنفس القدر تقريبًا، بافتراض ثبات الاتجاه العام.

In [ ]:
# Build the future business days for the next 2 months (weekdays only, consistent with the historical data)
last_date = dataframe['date_'].max()
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), end=last_date + pd.DateOffset(months=2))

future_df = pd.DataFrame({'date_': future_dates})
future_df['weekday'] = future_df['date_'].dt.dayofweek
future_df['t'] = np.arange(len(model_df), len(model_df) + len(future_df))

future_dummies = pd.get_dummies(future_df['weekday'], prefix='wd', drop_first=True)
future_dummies = future_dummies.reindex(columns=weekday_dummies.columns, fill_value=0)  # keep columns aligned with training
X_future = pd.concat([future_df[['t']], future_dummies], axis=1)

print(f"Forecast horizon: {future_dates.min().date()} -> {future_dates.max().date()}  ({len(future_dates)} business days)")

In [ ]:
# Predict the expected watch hours for the next 2 months
future_df['Total_watch_time_in_houres'] = model.predict(X_future)
future_df[['date_','weekday','Total_watch_time_in_houres']].head(10)

In [ ]:
# combine historical and forecasted data into a single dataframe for plotting
hist_plot = dataframe[['date_','Total_watch_time_in_houres']].copy()
hist_plot['type'] = 'Actual'
fc_plot = future_df[['date_','Total_watch_time_in_houres']].copy()
fc_plot['type'] = 'Forecast'
combined = pd.concat([hist_plot, fc_plot], ignore_index=True)

fig = px.line(combined, x='date_', y='Total_watch_time_in_houres', color='type',
              title='Daily total watch hours - actual vs forecast (next 2 months)',
              color_discrete_map={'Actual':'#5B8FF9','Forecast':'#E8684A'})
fig.show()

In [ ]:
# Identify the potential peak days within the forecast horizon (highest predicted watch hours)
top_peak_days = future_df.sort_values('Total_watch_time_in_houres', ascending=False).head(5).copy()
top_peak_days['weekday_name'] = top_peak_days['date_'].dt.day_name()
top_peak_days['Total_watch_time_in_houres'] = top_peak_days['Total_watch_time_in_houres'].round(1)
top_peak_days[['date_','weekday_name','Total_watch_time_in_houres']]

In [ ]:
# A simple weekly summary table - the easiest artifact for decision makers to read at a glance
future_df['week_start'] = future_df['date_'].dt.to_period('W').apply(lambda r: r.start_time)
weekly_forecast = future_df.groupby('week_start')['Total_watch_time_in_houres'].agg(
    avg_per_day='mean', total_hours='sum').round(1).reset_index()
weekly_forecast

In [ ]:
fig = px.bar(weekly_forecast, x='week_start', y='total_hours',
             title='Forecasted total watch hours per week (next 2 months)',
             labels={'week_start':'Week starting','total_hours':'Total watch hours'})
fig.show()

In [ ]:
# overall summary numbers useful for decision makers
total_forecast_hours = future_df['Total_watch_time_in_houres'].sum()
avg_forecast_per_day = future_df['Total_watch_time_in_houres'].mean()
last_month_hist_avg = dataframe[dataframe['date_'] >= dataframe['date_'].max() - pd.Timedelta(days=30)]['Total_watch_time_in_houres'].mean()
month1_avg = future_df['Total_watch_time_in_houres'].head(22).mean()
month2_avg = future_df['Total_watch_time_in_houres'].tail(22).mean()

print(f"Total predicted watch hours over the next 2 months: {total_forecast_hours:,.0f} hours")
print(f"Average predicted watch hours per day: {avg_forecast_per_day:,.1f} hours")
print(f"Historical average (last 30 days): {last_month_hist_avg:,.1f} hours/day")
print(f"Forecast month 1 average: {month1_avg:,.1f} hours/day")
print(f"Forecast month 2 average: {month2_avg:,.1f} hours/day")
print(f"Change vs last historical month: {(month1_avg/last_month_hist_avg - 1)*100:,.1f}%")
print(f"Change month 1 -> month 2 of the forecast: {(month2_avg/month1_avg - 1)*100:,.1f}%")

### نتائج بناء النموذج والتنبؤ
- **جودة النموذج**: على بيانات الاختبار (آخر 10 أيام عمل لم يرها النموذج)، حقق النموذج خطأ متوسط مطلق (MAE) ≈ 44 ساعة فقط، وهو خطأ معقول جدًا مقارنة بمتوسط المشاهدة اليومي (≈ 780 ساعة)، أي بنسبة خطأ نسبية أقل من 6%.
- **الاتجاه العام**: تُظهر البيانات اتجاهًا تنازليًا واضحًا في إجمالي ساعات المشاهدة طوال الفترة (من ≈1,123 ساعة في بداية يناير إلى ≈609 ساعة في نهاية أبريل)، ويستمر النموذج بعكس هذا الاتجاه في تنبؤاته للشهرين القادمين (مايو ويونيو).
- **التنبؤ للشهرين القادمين**: يتوقع النموذج إجمالي **≈25,222 ساعة مشاهدة** خلال 44 يوم عمل قادم، بمتوسط **≈573 ساعة/يوم**، بانخفاض نسبته ≈10% عن متوسط آخر 30 يومًا من البيانات التاريخية. كما يستمر التراجع داخل فترة التنبؤ نفسها (متوسط الشهر الأول من التنبؤ ≈608 ساعة/يوم مقابل ≈539 ساعة/يوم في الشهر الثاني، أي تراجع إضافي ≈11%).
- **أوقات الذروة المحتملة**:
  - على مستوى الأسبوع: يوم **الجمعة** هو الأعلى تاريخيًا من حيث متوسط ساعات المشاهدة، يليه يوم **الإثنين**، بينما **الأربعاء** هو الأدنى. يُتوقع أن يستمر هذا النمط الأسبوعي في فترة التنبؤ (أيام الجمعة والإثنين هي الأعلى ضمن كل أسبوع متوقع).
  - على مستوى الشهرين القادمين: بسبب الاتجاه التنازلي العام، فإن الذروة المتوقعة (أعلى أيام المشاهدة) تتركز في **الأسابيع الأولى من شهر مايو**، بينما يُتوقع أن تكون أقل نقاط المشاهدة في **نهاية يونيو**.

## الخلاصة والتوصيات
1. النموذج المستخدم (انحدار خطي بسيط يجمع بين الاتجاه الزمني وتأثير يوم الأسبوع) يوفر للمسؤولين أداة سهلة وسريعة لتقدير إجمالي ساعات المشاهدة المتوقعة لأي يوم أو أسبوع قادم، مع نسبة خطأ منخفضة نسبيًا.
2. الاتجاه التنازلي المستمر يستدعي تنبيه فرق المحتوى والتسويق لاتخاذ إجراءات لتحفيز المشاهدة (مثل حملات تسويقية أو إصدارات محتوى جديد) قبل دخول فترة الانخفاض المتوقعة في الشهرين القادمين.
3. التخطيط لصيانة الأنظمة أو إطلاق تحديثات تقنية يُفضّل أن يتم في أيام منخفضة الذروة المتوقعة (الأربعاء/الخميس)، بينما تخصيص أكبر سعة (Capacity) لخوادم البث يجب أن يتركّز في أيام الذروة (الجمعة والإثنين)، خصوصًا في الأسابيع الأولى من فترة التنبؤ.
4. يُنصح بإعادة تدريب النموذج بشكل دوري (شهريًا مثلًا) كلما توفرت بيانات فعلية جديدة، لضمان مواكبة التنبؤ لأي تغيّر مستقبلي في نمط السلوك لم يكن ظاهرًا في بيانات يناير-أبريل.